# Kaggle Mamba Training Notebook

Designed for **Save Version → Run All** (commit run). Do not run interactively.

Mamba's sequential scan is ~25× slower than the transformer on T4. Training to 50k steps takes ~23h total across **3 Kaggle sessions** of ~8h each (12h commit run limit).

**The resume logic is automatic** — re-commit this notebook each session and it picks up from the last checkpoint.

| Session | Expected start step | Expected end step |
|---------|--------------------|-----------------|
| 1 | 0 | ~16,000 |
| 2 | ~16,000 | ~32,000 |
| 3 | ~32,000 | 50,000 |

**Before each commit:**
- Accelerator → GPU T4 x1
- Internet ON
- Persistence ON
- If resuming: attach the previous session's output dataset (see Cell 3b)

In [ ]:
# Cell 1: Clone repo and set working directory
import os, subprocess, sys

if not os.path.exists('transformer-vs-ssm'):
    subprocess.run(['git', 'clone',
                    'https://github.com/nvaidyan1/transformer-vs-ssm.git'], check=True)

os.chdir('transformer-vs-ssm')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f'Working directory: {os.getcwd()}')

In [ ]:
# Cell 2: Install dependencies and pull latest
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'torch', 'numpy', 'pyyaml', 'matplotlib', 'seaborn', 'tqdm'],
    check=True
)
print('deps OK')
subprocess.run(['git', 'pull'], check=True)

In [ ]:
# Cell 3a: Prepare data (skipped if already done)
from src.data import prepare_data
prepare_data()

In [ ]:
# Cell 3b: Restore checkpoint from previous session (sessions 2 and 3 only)
#
# On session 1: leave this cell as-is (the if-block will be skipped).
# On sessions 2+: attach your previous output as a Kaggle dataset, then set:
#   PREV_DATASET = '/kaggle/input/<your-dataset-name>'
#
# The dataset must contain checkpoints/mamba/ckpt_XXXXXXX.pt from the previous run.

import shutil
from pathlib import Path

PREV_DATASET = None  # e.g. '/kaggle/input/mamba-session-1'

if PREV_DATASET and Path(PREV_DATASET).exists():
    src = Path(PREV_DATASET) / 'checkpoints' / 'mamba'
    dst = Path('checkpoints/mamba')
    dst.mkdir(parents=True, exist_ok=True)
    for pt in sorted(src.glob('ckpt_???????.pt')):
        shutil.copy2(pt, dst / pt.name)
        print(f'Restored: {pt.name}')
    # Recreate latest.pt symlink pointing to the highest-numbered checkpoint
    all_pts = sorted(dst.glob('ckpt_???????.pt'))
    if all_pts:
        latest = dst / 'latest.pt'
        if latest.is_symlink() or latest.exists():
            latest.unlink()
        latest.symlink_to(all_pts[-1].name)
        print(f'latest.pt → {all_pts[-1].name}')
else:
    print('Session 1 (no previous checkpoint to restore)')

In [ ]:
# Cell 4: Run checkpoint safety tests
result = subprocess.run(
    [sys.executable, 'tests/test_checkpoint_fixes.py'],
    capture_output=True, text=True
)
print(result.stdout[-3000:])
if result.returncode != 0:
    print(result.stderr[-1000:])
assert result.returncode == 0, 'Tests failed — not starting training'

In [ ]:
# Cell 5: Verify disk and show resume state
import shutil, torch
from pathlib import Path

free_gb = shutil.disk_usage('/kaggle/working').free / 1024**3
print(f'Free disk: {free_gb:.1f} GB')
assert free_gb > 4.0, f'Need ≥4 GB free, got {free_gb:.1f} GB'

latest = Path('checkpoints/mamba/latest.pt')
if latest.exists():
    ckpt = torch.load(latest, map_location='cpu', weights_only=False)
    print(f'Will resume from step {ckpt["step"]:,} (val_bpc {ckpt["val_bpc"]:.4f})')
else:
    print('No checkpoint found — starting from step 0')

In [ ]:
# Cell 6: Train Mamba (~8h per session, resumes automatically)
# -u: unbuffered stdout so progress lines stream to the output panel
print('=== MAMBA TRAINING START ===')
result = subprocess.run(
    [sys.executable, '-u', 'src/train.py', '--config', 'configs/mamba.yaml'],
    check=False
)
print(f'=== MAMBA EXIT CODE: {result.returncode} ===')
# A non-zero exit here means the run hit the session time limit (expected for sessions 1 and 2)
# Only assert on session 3 where we expect the full 50k steps

In [ ]:
# Cell 7: Zip checkpoint for this session (always run — even if training hit the time limit)
import zipfile
from pathlib import Path

latest = Path('checkpoints/mamba/latest.pt')
if latest.exists():
    ckpt = torch.load(latest, map_location='cpu', weights_only=False)
    step = ckpt['step']
    bpc  = ckpt['val_bpc']
    print(f'Checkpoint: step={step:,}  val_bpc={bpc:.4f}')
else:
    print('WARNING: no checkpoint found — training may have failed before the first save')
    step = 0

zip_path = Path(f'/kaggle/working/mamba_checkpoint_step{step:07d}.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(Path('checkpoints/mamba').rglob('*.pt')):
        zf.write(f)

size_mb = zip_path.stat().st_size / 1024**2
print(f'Zipped → {zip_path} ({size_mb:.1f} MB)')
print('Download via: notebook Output tab')
print('On next session: attach this output as a Kaggle dataset and set PREV_DATASET in Cell 3b')